## Parado = TRUE

In [46]:
df = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/train_final.parquet")

df_meteo = pd.read_csv("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/datos_meteorologicos.csv", delimiter=",")

In [48]:
df["llegada_punto"]  = pd.to_datetime(df["llegada_punto"])
df["salida_punto"]   = pd.to_datetime(df["salida_punto"])
df["despegue"]       = pd.to_datetime(df["despegue"])
df["timestamp"]      = pd.to_datetime(df["timestamp"])
df["fecha_despegue"] = pd.to_datetime(df["fecha_despegue"])

In [49]:
df_parado = df[df["parado"]].copy()

In [50]:
# Convertir comas a puntos y strings a float donde sea necesario
cols_numericas = [
    "Precipitación", "Temperatura", "Humedad", "Viento", "Viento máximo",
    "Temperatura mínima", "Temperatura máxima"
]
for col in cols_numericas:
    df_meteo[col] = df_meteo[col].str.replace(",", ".").astype(float)

# Convertir 'Fecha' a datetime.date y 'Hora' a número
df_meteo["Fecha"] = pd.to_datetime(df_meteo["Fecha"]).dt.date
df_meteo["Hora"] = pd.to_datetime(df_meteo["Hora"], format="%H:%M").dt.hour

In [51]:
# Creamos las columnas necesarias en df_parado
df_parado["Fecha"] = df_parado["timestamp"].dt.date
df_parado["Hora"] = df_parado["timestamp"].dt.hour

# Hacemos el merge
df_merged = df_parado.merge(df_meteo, how="left", on=["Fecha", "Hora"])

In [52]:
df_filtrado = df_merged[df_merged["tiempo_espera"] <= 500].copy()

print(f"Registros antes de filtrar: {len(df_merged)}")
print(f"Registros después de filtrar: {len(df_filtrado)}")

Registros antes de filtrar: 171015
Registros después de filtrar: 165447


In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.notebook import tqdm

df_modelo = df_filtrado.drop(columns=[
    "ICAO", "llegada_punto", "salida_punto", "salida_lon", "salida_lat","despegue",
    "runway", "fecha_despegue", "hora_despegue", "timestamp",
    "holding_point" 
])

In [54]:
X = df_modelo.drop(columns=["tiempo_espera"])
y = df_modelo["tiempo_espera"]

In [55]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object","bool"]).columns.tolist()

In [56]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

preprocessor.fit(X)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['llegada_lon', 'llegada_lat',
                                  'tiempo_esperado', 'runway_occupied',
                                  'queue_length', 'time_since_free',
                                  'Precipitación', 'Temperatura', 'Humedad',
                                  'Viento', 'Viento máximo',
                                  'Temperatura mínima', 'Temperatura máxima']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['aircraft_type', 'parado', 'Fecha',
                                  'Dirección', 'Dirección viento máximo'])])

In [57]:
cat_feats = preprocessor.named_transformers_["cat"]\
    .get_feature_names_out(cat_cols).tolist()
all_feats = num_cols + cat_feats + ["otros_cluster_ocupados", "tiempo_esperado"]
print("Features que entran al modelo:")
for f in all_feats:
    print(" ",f)

Features que entran al modelo:
  llegada_lon
  llegada_lat
  tiempo_esperado
  runway_occupied
  queue_length
  time_since_free
  Precipitación
  Temperatura
  Humedad
  Viento
  Viento máximo
  Temperatura mínima
  Temperatura máxima
  aircraft_type_Heavy (larger than 136000 kg)
  aircraft_type_High vortex aircraft
  aircraft_type_Light (less than 7000 kg)
  aircraft_type_Medium 1 (between 7000 kg and 34000 kg)
  aircraft_type_Medium 2 (between 34000 kg to 136000 kg)
  aircraft_type_None
  parado_True
  Fecha_2024-11-07
  Fecha_2024-11-08
  Fecha_2024-11-09
  Fecha_2024-11-10
  Fecha_2024-11-11
  Fecha_2024-11-12
  Fecha_2024-11-13
  Fecha_2024-11-14
  Fecha_2024-11-15
  Fecha_2024-11-16
  Fecha_2024-11-17
  Fecha_2024-11-18
  Fecha_2024-11-19
  Fecha_2024-11-20
  Fecha_2024-11-21
  Fecha_2024-11-22
  Fecha_2024-11-23
  Fecha_2024-11-24
  Fecha_2024-11-25
  Fecha_2024-11-26
  Fecha_2024-11-27
  Fecha_2024-11-28
  Fecha_2024-11-30
  Fecha_2024-12-02
  Fecha_2024-12-03
  Fecha_2024-12-0

In [58]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [61]:
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)

n_estimators = 100
rf = RandomForestRegressor(n_estimators=1, warm_start=True, random_state=42)

for i in tqdm(range(1, n_estimators + 1), desc="Entrenando RF"):
    rf.n_estimators = i
    rf.fit(X_train_t, y_train)

y_pred = rf.predict(X_test_t)

Entrenando RF:   0%|          | 0/100 [00:00<?, ?it/s]

In [62]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("🔹 Random Forest final con congestión entre puntos:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

🔹 Random Forest final con congestión entre puntos:
MAE: 14.79
RMSE: 25.70
R²: 0.919


In [66]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1) Cargo test_final
df_test = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/test_final.parquet")

# 2) Solo vuelos en holding
df_test_parado = df_test[df_test["parado"]].copy()

# 3) Creo Fecha y Hora para el merge con meteorología
df_test_parado["Fecha"] = df_test_parado["timestamp"].dt.date
df_test_parado["Hora"]  = df_test_parado["timestamp"].dt.hour

# 4) Merge con df_meteo (ya procesado antes)
df_test_merged = df_test_parado.merge(df_meteo, on=["Fecha","Hora"], how="left")

# 5) Filtrar outliers igual que en train
df_test_filtrado = df_test_merged[df_test_merged["tiempo_espera"] <= 500].copy()

# 6) Preparo X_test_final e y_test_final
drop_cols = [
    "ICAO","llegada_punto","salida_punto","despegue",
    "runway","fecha_despegue","hora_despegue",
    "timestamp","holding_point"
]
X_test_final = df_test_filtrado.drop(columns=drop_cols + ["tiempo_espera"])
y_test_final = df_test_filtrado["tiempo_espera"]

# 7) Predicción y MAE
from sklearn.metrics import mean_absolute_error

# 7a) Transformar X_test_final con el preprocessor entrenado
X_test_t = preprocessor.transform(X_test_final)

# 7b) Predecir con tu rf (que ya está entrenado)
y_pred_test = rf.predict(X_test_t)

df_output = df_test_filtrado.copy()
df_output['pred'] = y_pred_test

# 7c) Calcular MAE
mae_test = mean_absolute_error(y_test_final, y_pred_test)
print(f"MAE en test_final: {mae_test:.2f} segundos")


MAE en test_final: 14.82 segundos


In [67]:
df_output.to_csv('test_final_with_pred.csv', index=False)

print("CSV generado: test_final_with_pred.csv")

CSV generado: test_final_with_pred.csv


## Parado = False

In [90]:
df = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/train_final.parquet")

df_meteo = pd.read_csv("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/datos_meteorologicos.csv", delimiter=",")

In [91]:
df["llegada_punto"]  = pd.to_datetime(df["llegada_punto"])
df["salida_punto"]   = pd.to_datetime(df["salida_punto"])
df["despegue"]       = pd.to_datetime(df["despegue"])
df["timestamp"]      = pd.to_datetime(df["timestamp"])
df["fecha_despegue"] = pd.to_datetime(df["fecha_despegue"])

In [92]:
df_no_parado = df[df["parado"] == False].copy()

In [93]:
# Convertir comas a puntos y strings a float donde sea necesario
cols_numericas = [
    "Precipitación", "Temperatura", "Humedad", "Viento", "Viento máximo",
    "Temperatura mínima", "Temperatura máxima"
]
for col in cols_numericas:
    df_meteo[col] = df_meteo[col].str.replace(",", ".").astype(float)

# Convertir 'Fecha' a datetime.date y 'Hora' a número
df_meteo["Fecha"] = pd.to_datetime(df_meteo["Fecha"]).dt.date
df_meteo["Hora"] = pd.to_datetime(df_meteo["Hora"], format="%H:%M").dt.hour

In [94]:
# Creamos las columnas necesarias en df_parado
df_no_parado["Fecha"] = df_no_parado["timestamp"].dt.date
df_no_parado["Hora"] = df_no_parado["timestamp"].dt.hour

# Hacemos el merge
df_merged = df_no_parado.merge(df_meteo, how="left", on=["Fecha", "Hora"])

In [95]:
df_filtrado = df_merged[df_merged["tiempo_espera"] <= 500].copy()

print(f"Registros antes de filtrar: {len(df_merged)}")
print(f"Registros después de filtrar: {len(df_filtrado)}")

Registros antes de filtrar: 87837
Registros después de filtrar: 86690


In [96]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.notebook import tqdm

df_modelo = df_filtrado.drop(columns=[
    "ICAO", "llegada_punto", "salida_punto", "salida_lon", "salida_lat","despegue",
    "runway", "fecha_despegue", "hora_despegue", "timestamp",
    "holding_point" 
])

In [97]:
X = df_modelo.drop(columns=["tiempo_espera"])
y = df_modelo["tiempo_espera"]

In [98]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object","bool"]).columns.tolist()

In [99]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

preprocessor.fit(X)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['llegada_lon', 'llegada_lat',
                                  'tiempo_esperado', 'runway_occupied',
                                  'queue_length', 'time_since_free',
                                  'hold_pt_occupied', 'Precipitación',
                                  'Temperatura', 'Humedad', 'Viento',
                                  'Viento máximo', 'Temperatura mínima',
                                  'Temperatura máxima']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['aircraft_type', 'parado', 'Fecha',
                                  'Dirección', 'Dirección viento máximo'])])

In [100]:
cat_feats = preprocessor.named_transformers_["cat"]\
    .get_feature_names_out(cat_cols).tolist()
all_feats = num_cols + cat_feats + ["otros_cluster_ocupados", "tiempo_esperado"]
print("Features que entran al modelo:")
for f in all_feats:
    print(" ",f)

Features que entran al modelo:
  llegada_lon
  llegada_lat
  tiempo_esperado
  runway_occupied
  queue_length
  time_since_free
  hold_pt_occupied
  Precipitación
  Temperatura
  Humedad
  Viento
  Viento máximo
  Temperatura mínima
  Temperatura máxima
  aircraft_type_Heavy (larger than 136000 kg)
  aircraft_type_High vortex aircraft
  aircraft_type_Light (less than 7000 kg)
  aircraft_type_Medium 1 (between 7000 kg and 34000 kg)
  aircraft_type_Medium 2 (between 34000 kg to 136000 kg)
  aircraft_type_None
  parado_False
  Fecha_2024-11-07
  Fecha_2024-11-08
  Fecha_2024-11-09
  Fecha_2024-11-10
  Fecha_2024-11-11
  Fecha_2024-11-12
  Fecha_2024-11-13
  Fecha_2024-11-14
  Fecha_2024-11-15
  Fecha_2024-11-16
  Fecha_2024-11-17
  Fecha_2024-11-18
  Fecha_2024-11-19
  Fecha_2024-11-20
  Fecha_2024-11-21
  Fecha_2024-11-22
  Fecha_2024-11-23
  Fecha_2024-11-24
  Fecha_2024-11-25
  Fecha_2024-11-26
  Fecha_2024-11-27
  Fecha_2024-11-28
  Fecha_2024-11-30
  Fecha_2024-12-02
  Fecha_2024-12-

In [101]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

In [103]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [104]:
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)

n_estimators = 100
rf = RandomForestRegressor(n_estimators=1, warm_start=True, random_state=42)

for i in tqdm(range(1, n_estimators + 1), desc="Entrenando RF"):
    rf.n_estimators = i
    rf.fit(X_train_t, y_train)

y_pred = rf.predict(X_test_t)

Entrenando RF:   0%|          | 0/100 [00:00<?, ?it/s]

In [105]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("🔹 Random Forest final con congestión entre puntos:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

🔹 Random Forest final con congestión entre puntos:
MAE: 20.33
RMSE: 33.04
R²: 0.816


In [108]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1) Cargo test_final
df_test = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/test_final.parquet")

# 2) Solo vuelos en holding
df_test_no_parado = df[df["parado"] == False].copy()

# 3) Creo Fecha y Hora para el merge con meteorología
df_test_no_parado["Fecha"] = df_test_no_parado["timestamp"].dt.date
df_test_no_parado["Hora"]  = df_test_no_parado["timestamp"].dt.hour

# 4) Merge con df_meteo (ya procesado antes)
df_test_merged = df_test_no_parado.merge(df_meteo, on=["Fecha","Hora"], how="left")

# 5) Filtrar outliers igual que en train
df_test_filtrado = df_test_merged[df_test_merged["tiempo_espera"] <= 500].copy()

# 6) Preparo X_test_final e y_test_final
drop_cols = [
    "ICAO","llegada_punto","salida_punto","despegue",
    "runway","fecha_despegue","hora_despegue",
    "timestamp","holding_point"
]
X_test_final = df_test_filtrado.drop(columns=drop_cols + ["tiempo_espera"])
y_test_final = df_test_filtrado["tiempo_espera"]

# 7) Predicción y MAE
from sklearn.metrics import mean_absolute_error

# 7a) Transformar X_test_final con el preprocessor entrenado
X_test_t = preprocessor.transform(X_test_final)

# 7b) Predecir con tu rf (que ya está entrenado)
y_pred_test = rf.predict(X_test_t)

df_output = df_test_filtrado.copy()
df_output['pred'] = y_pred_test

# 7c) Calcular MAE
mae_test = mean_absolute_error(y_test_final, y_pred_test)
print(f"MAE en test_final: {mae_test:.2f} segundos")

MAE en test_final: 10.12 segundos


In [109]:
df_output.to_csv('test_final_with_pred_no_parado.csv', index=False)

print("CSV generado: test_final_with_pred_no_parado.csv")

CSV generado: test_final_with_pred_no_parado.csv


## Parado = True y False

In [3]:
df = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/train_final.parquet")

df_meteo = pd.read_csv("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/datos_meteorologicos.csv", delimiter=",")

In [4]:
df["llegada_punto"]  = pd.to_datetime(df["llegada_punto"])
df["salida_punto"]   = pd.to_datetime(df["salida_punto"])
df["despegue"]       = pd.to_datetime(df["despegue"])
df["timestamp"]      = pd.to_datetime(df["timestamp"])
df["fecha_despegue"] = pd.to_datetime(df["fecha_despegue"])

In [5]:
# Convertir comas a puntos y strings a float donde sea necesario
cols_numericas = [
    "Precipitación", "Temperatura", "Humedad", "Viento", "Viento máximo",
    "Temperatura mínima", "Temperatura máxima"
]
for col in cols_numericas:
    df_meteo[col] = df_meteo[col].str.replace(",", ".").astype(float)

# Convertir 'Fecha' a datetime.date y 'Hora' a número
df_meteo["Fecha"] = pd.to_datetime(df_meteo["Fecha"]).dt.date
df_meteo["Hora"] = pd.to_datetime(df_meteo["Hora"], format="%H:%M").dt.hour

In [6]:
# Creamos las columnas necesarias en df_parado
df["Fecha"] = df["timestamp"].dt.date
df["Hora"] = df["timestamp"].dt.hour

# Hacemos el merge
df_merged = df.merge(df_meteo, how="left", on=["Fecha", "Hora"])

In [7]:
df_filtrado = df_merged[df_merged["tiempo_espera"] <= 500].copy()

print(f"Registros antes de filtrar: {len(df_merged)}")
print(f"Registros después de filtrar: {len(df_filtrado)}")

Registros antes de filtrar: 258852
Registros después de filtrar: 252137


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.notebook import tqdm

df_modelo = df_filtrado.drop(columns=[
    "ICAO", "llegada_punto", "salida_punto", "salida_lon", "salida_lat","despegue",
    "runway", "fecha_despegue", "hora_despegue", "timestamp",
    "holding_point" 
])

In [9]:
X = df_modelo.drop(columns=["tiempo_espera"])
y = df_modelo["tiempo_espera"]

In [10]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object","bool"]).columns.tolist()

In [11]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

preprocessor.fit(X)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['llegada_lon', 'llegada_lat',
                                  'tiempo_esperado', 'runway_occupied',
                                  'queue_length', 'time_since_free',
                                  'hold_pt_occupied', 'Precipitación',
                                  'Temperatura', 'Humedad', 'Viento',
                                  'Viento máximo', 'Temperatura mínima',
                                  'Temperatura máxima']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['aircraft_type', 'parado', 'Fecha',
                                  'Dirección', 'Dirección viento máximo'])])

In [12]:
cat_feats = preprocessor.named_transformers_["cat"]\
    .get_feature_names_out(cat_cols).tolist()
all_feats = num_cols + cat_feats + ["otros_cluster_ocupados", "tiempo_esperado"]
print("Features que entran al modelo:")
for f in all_feats:
    print(" ",f)

Features que entran al modelo:
  llegada_lon
  llegada_lat
  tiempo_esperado
  runway_occupied
  queue_length
  time_since_free
  hold_pt_occupied
  Precipitación
  Temperatura
  Humedad
  Viento
  Viento máximo
  Temperatura mínima
  Temperatura máxima
  aircraft_type_Heavy (larger than 136000 kg)
  aircraft_type_High vortex aircraft
  aircraft_type_Light (less than 7000 kg)
  aircraft_type_Medium 1 (between 7000 kg and 34000 kg)
  aircraft_type_Medium 2 (between 34000 kg to 136000 kg)
  aircraft_type_None
  parado_False
  parado_True
  Fecha_2024-11-07
  Fecha_2024-11-08
  Fecha_2024-11-09
  Fecha_2024-11-10
  Fecha_2024-11-11
  Fecha_2024-11-12
  Fecha_2024-11-13
  Fecha_2024-11-14
  Fecha_2024-11-15
  Fecha_2024-11-16
  Fecha_2024-11-17
  Fecha_2024-11-18
  Fecha_2024-11-19
  Fecha_2024-11-20
  Fecha_2024-11-21
  Fecha_2024-11-22
  Fecha_2024-11-23
  Fecha_2024-11-24
  Fecha_2024-11-25
  Fecha_2024-11-26
  Fecha_2024-11-27
  Fecha_2024-11-28
  Fecha_2024-11-30
  Fecha_2024-12-02
  

In [13]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)

n_estimators = 100
rf = RandomForestRegressor(n_estimators=1, warm_start=True, random_state=42)

for i in tqdm(range(1, n_estimators + 1), desc="Entrenando RF"):
    rf.n_estimators = i
    rf.fit(X_train_t, y_train)

y_pred = rf.predict(X_test_t)

Entrenando RF:   0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("🔹 Random Forest final con congestión entre puntos:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

🔹 Random Forest final con congestión entre puntos:
MAE: 16.73
RMSE: 28.72
R²: 0.906


In [20]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1) Cargo test_final
df_test = pd.read_parquet("C:/Users/jesus/Escritorio/Uni/PD2/PD2/data/Train/test_final.parquet")

# 3) Creo Fecha y Hora para el merge con meteorología
df["Fecha"] = df["timestamp"].dt.date
df["Hora"]  = df["timestamp"].dt.hour

# 4) Merge con df_meteo (ya procesado antes)
df_test_merged = df.merge(df_meteo, on=["Fecha","Hora"], how="left")

# 6) Preparo X_test_final e y_test_final
drop_cols = [
    "ICAO","llegada_punto","salida_punto","despegue",
    "runway","fecha_despegue","hora_despegue",
    "timestamp","holding_point"
]
X_test_final = df_test_merged.drop(columns=drop_cols + ["tiempo_espera"])
y_test_final = df_test_merged["tiempo_espera"]

# 7) Predicción y MAE
from sklearn.metrics import mean_absolute_error

# 7a) Transformar X_test_final con el preprocessor entrenado
X_test_t = preprocessor.transform(X_test_final)

# 7b) Predecir con tu rf (que ya está entrenado)
y_pred_test = rf.predict(X_test_t)

df_output = df_test_merged.copy()
df_output['pred'] = y_pred_test

# 7c) Calcular MAE
mae_test = mean_absolute_error(y_test_final, y_pred_test)
print(f"MAE en test_final: {mae_test:.2f} segundos")

MAE en test_final: 16.56 segundos


In [21]:
df_output.to_csv('test_final_with_pred_arreglado.csv', index=False)

print("CSV generado: test_final_with_pred_arreglado.csv")

CSV generado: test_final_with_pred_arreglado.csv


## Parado = True y False Entrenando con todo train

In [3]:
df_train = pd.read_parquet("../../../data/Train/train_final.parquet")
df_test = pd.read_parquet("../../../data/Train/test_final.parquet")
df_train.to_csv('train.csv', index=False)
df_test.to_csv('test.csv', index=False)

In [4]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# Cargar datos
df_train = pd.read_parquet("../../../data/Train/train_final.parquet")
df_test = pd.read_parquet("../../../data/Train/test_final.parquet")
df_meteo = pd.read_csv("../../../data/datos_meteorologicos.csv", delimiter=",")

# Procesamiento fechas
for df in [df_train, df_test]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["Fecha"] = df["timestamp"].dt.date
    df["Hora"] = df["timestamp"].dt.hour

df_meteo["Fecha"] = pd.to_datetime(df_meteo["Fecha"]).dt.date
df_meteo["Hora"] = pd.to_datetime(df_meteo["Hora"], format="%H:%M").dt.hour
cols_numericas = ["Precipitación", "Temperatura", "Humedad", "Viento", "Viento máximo", "Temperatura mínima", "Temperatura máxima"]
for col in cols_numericas:
    df_meteo[col] = df_meteo[col].str.replace(",", ".").astype(float)

# Merge y filtrado
df_train = df_train.merge(df_meteo, how="left", on=["Fecha", "Hora"])
df_test = df_test.merge(df_meteo, how="left", on=["Fecha", "Hora"])

#df_train = df_train[df_train["tiempo_espera"] <= 500].copy()

# Columnas a eliminar
drop_cols = ["ICAO", "llegada_punto", "salida_punto", "salida_lon", "salida_lat", "despegue", "runway", "fecha_despegue", "hora_despegue", "timestamp", "holding_point"]
X_train = df_train.drop(columns=drop_cols + ["tiempo_espera"])
y_train = df_train["tiempo_espera"]
X_test = df_test.drop(columns=drop_cols + ["tiempo_espera"])
y_test = df_test["tiempo_espera"]

# Preprocesamiento
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object","bool"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

# Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

# GridSearchCV (ligero para ir rápido, puedes expandirlo luego)
param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [None, 10],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring="neg_mean_absolute_error", n_jobs=-1, verbose=10)
grid_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid_search.best_params_)

# Predicción
y_pred = grid_search.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE en test_final: {mae:.2f} segundos")


Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 2/3; 1/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=50
[CV 3/3; 1/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=50
[CV 1/3; 1/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=50
[CV 1/3; 2/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=100
[CV 2/3; 2/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=100
[CV 3/3; 2/8] START model__max_depth=None, model__min_samples_split=2, model__n_estimators=100
[CV 1/3; 3/8] START model__max_depth=None, model__min_samples_split=5, model__n_estimators=50
[CV 2/3; 3/8] START model__max_depth=None, model__min_samples_split=5, model__n_estimators=50
[CV 1/3; 3/8] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=50;, score=-21.893 total time=11.2min
[CV 3/3; 3/8] START model__max_depth=None, model__min_samp

In [6]:
# Guardar predicciones
df_output = df_test.copy()
df_output["pred"] = y_pred
#df_output = df_output[df_output.tiempo_espera < 500] # para analizar solo menores que 500
df_output.to_csv("predicciones_modelo_sklearn.csv", index=False)
print("CSV generado: predicciones_modelo_sklearn.csv")

CSV generado: test_final_with_pred_arreglado_2.csv
